# Legal Information Extraction Pipeline — Gemma-2-9B-IT

Extracts structured legal information from Indian court judgments using **Google Gemma-2-9B-IT**.

**Output schema per judgment:**
```json
{ "case_type", "subject", "object", "objective_aspect", "subjective_aspect", "reasoning" }
```

**Note:** Gemma-2-9B-IT uses ~18 GB GPU memory in fp16. Ensure your GPU (24 GB 3090) is clean before loading — run `nvidia-smi` and kill stray processes first.

## 1. Package Installation

In [1]:
# Run once, then RESTART the kernel.
# Gemma-2 needs transformers >= 4.42. Pin 4.46.3 for stability on Python 3.9.
%pip install -q -U "transformers==4.46.3" "tokenizers>=0.20,<0.21" \
    "accelerate>=0.26" "huggingface_hub>=0.24" sentencepiece pandas openpyxl tqdm

    torch (>=1.7.*)
           ~~~~~~^
Note: you may need to restart the kernel to use updated packages.


### (Optional) Check GPU Memory

In [3]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

Wed Jun 24 17:51:36 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 530.30.02              Driver Version: 530.30.02    CUDA Version: 12.1     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                  Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf            Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090         On | 00000000:3B:00.0 Off |                  N/A |
| 51%   55C    P8               26W / 350W|      8MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
!kill 14073 17640 14073

## 2. Imports

In [4]:
import os, re, json, time, traceback
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

import transformers
from packaging import version
print("transformers:", transformers.__version__)
assert version.parse(transformers.__version__) >= version.parse("4.42"), (
    "Gemma-2 needs transformers >= 4.42. Run the install cell above, then RESTART the kernel."
)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

/mnt/Data/yashv7523/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/mnt/Data/yashv7523/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


transformers: 4.46.3
torch: 2.6.0+cu124
CUDA available: True


## 3. Configuration

Gemma-2-9B-IT specific settings. `SAMPLE_SIZE = 4` as requested.

In [5]:
# ----------------------------- MODEL -----------------------------
MODEL_NAME = "google/gemma-2-9b-it"

# Other models (uncomment to switch):
# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

# ----------------------------- DATA ------------------------------
INPUT_PATH   = "Allahabad_criminal_merged.xlsx"
OUTPUT_PATH  = "indilex_extracted_output_Gemma.xlsx"
TEXT_COLUMN  = "judgment_text"

# ----------------------------- RUN MODE --------------------------
SAMPLE_MODE  = True
SAMPLE_SIZE  = 4      # First 4 judgments as requested

# ----------------------------- GENERATION ------------------------
# Gemma-2 tuning:
#   - temperature=0.15 (low for structured extraction, slight randomness)
#   - top_p=0.9 (Gemma handles this well, not as verbose as DeepSeek)
#   - max_new_tokens=700 (Gemma is concise, no <think> overhead)
MAX_NEW_TOKENS   = 700
TEMPERATURE      = 0.15
TOP_P            = 0.9
DO_SAMPLE        = True
MAX_INPUT_TOKENS = 7000
MAX_RETRIES      = 3

# ----------------------------- 4E SCHEMA -------------------------
SCHEMA_KEYS = ["case_type", "subject", "object",
               "objective_aspect", "subjective_aspect", "reasoning"]
CASE_TYPES  = ["Criminal", "Civil", "Constitutional", "Administrative"]

## 4. Model Loading

Gemma-2-9B-IT needs ~18 GB fp16. The cell picks the first GPU with enough free memory and pins the model to it.

In [4]:
# Hugging Face login — required for gated models like Gemma
!huggingface-cli login --token "YOUR_HF_TOKEN_HERE"

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `testing` has been saved to /mnt/Data/yashv7523/.cache/huggingface/stored_tokens
Your token has been saved to /mnt/Data/yashv7523/.cache/huggingface/token
Login successful.
The current active token is: `testing`


In [6]:
import torch, gc

# --- Free any leftover GPU memory ---
for _v in ("model", "tokenizer"):
    if _v in globals():
        del globals()[_v]
gc.collect()
torch.cuda.empty_cache()

# --- GPU memory check ---
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f"GPUs visible: {n}")
    for d in range(n):
        free, total = torch.cuda.mem_get_info(d)
        print(f"  cuda:{d}  free {free/1e9:5.1f} GB / total {total/1e9:5.1f} GB")
    # Gemma-2-9B fp16 needs ~18 GB
    need_gb = 18
    pick = None
    for d in range(n):
        free, _ = torch.cuda.mem_get_info(d)
        if free/1e9 >= need_gb:
            pick = d
            break
    if pick is None:
        raise RuntimeError(
            f"No single GPU has ~{need_gb} GB free for Gemma-2-9B fp16. "
            "Free GPU memory first: run `nvidia-smi` in terminal, "
            "`kill -9 <PID>` stray python processes, then re-run this cell."
        )
    print(f"Loading on cuda:{pick}")
else:
    pick = None
    print("WARNING: CUDA not available -> CPU (very slow for 9B model).")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

try:
    if pick is not None:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        ).to(f"cuda:{pick}")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
except torch.cuda.OutOfMemoryError as e:
    torch.cuda.empty_cache()
    raise RuntimeError(
        "CUDA OOM while loading Gemma-2-9B. The model needs ~18 GB fp16. "
        "Kill stray GPU processes, RESTART kernel, and retry."
    ) from e
except (ValueError, KeyError) as e:
    raise RuntimeError(
        f"Failed to load {MODEL_NAME}. Likely transformers too old. "
        f"You have {transformers.__version__}; Gemma-2 needs >= 4.42."
    ) from e

model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded on:", next(model.parameters()).device)
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(pick)
    print(f"GPU memory after load: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

GPUs visible: 2
  cuda:0  free  24.6 GB / total  25.4 GB
  cuda:1  free  24.6 GB / total  25.4 GB
Loading on cuda:0


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded on: cuda:0
GPU memory after load: 6.1 GB free / 25.4 GB total


## 5. Dataset Loading

In [7]:
df = pd.read_excel(INPUT_PATH)

if TEXT_COLUMN not in df.columns:
    raise ValueError(f"'{TEXT_COLUMN}' not found. Available columns: {list(df.columns)}")

if "case_id" not in df.columns:
    df["case_id"] = df.index.astype(str)
if "language" not in df.columns:
    df["language"] = "unknown"

print("Full dataset shape:", df.shape)

work_df = (df.head(SAMPLE_SIZE).copy() if SAMPLE_MODE else df.copy())
print(f"Mode: {'SAMPLE' if SAMPLE_MODE else 'FULL'}  ->  processing {len(work_df)} records")
work_df.head()

Full dataset shape: (199, 10)
Mode: SAMPLE  ->  processing 4 records


,case_id,language,case_type,judgment_text,subject,object,objective_aspect,subjective_aspect,reasoning,legal provision
0,ALL_CRIM_000001,Hindi,Criminal,Court No.=27\n\nAPPLICATION UIS 482 No. - 741 ...,Dinesh Yadav @ Dinesh Kumar And 7 Others,Wife/Complainant,Assault; Bigamy; Cruelty; Intentional Insult; ...,Dowry Demand,NaN,NaN
1,ALL_CRIM_000002,Hindi,Criminal,‘Neutral Citation No. - 2023:AHC:145371\n\nCou...,Kamlesh Jaiswal Alias Monu Jaiswal And Another,State/Society,Criminal Law Amendment Act Offence,Not Clearly Specified,NaN,NaN
2,ALL_CRIM_000003,Hindi,Criminal,‘Neutral Citation No. - 2024:AHC-LKO:1629\nCou...,Bahal Khan And 4 Others,Complainant/Informant,Rioting; Rioting With Deadly Weapon; Assault; ...,Intent/Knowledge To Cause Death Or Serious Harm,NaN,NaN
3,ALL_CRIM_000004,Hindi,Criminal,"Court No, -27\n(Case := APPLICATION U/S 482 No...",Banshi Lal,Complainant/Informant,Kidnapping Or Abduction,Intent To Compel Wrongful Restraint/Confinement,NaN,NaN


## 6. Prompt Construction

Gemma-2 is well-behaved for structured output — it doesn't produce `<think>` blocks like DeepSeek, but can occasionally add brief explanations. The prompt is firm but not as aggressive as the DeepSeek version.

In [8]:
SYSTEM_PROMPT = (
    "You are an expert Indian legal analyst. You read court judgments and extract "
    "structured legal information. Return ONLY one valid JSON object. "
    "Do not explain your reasoning process. Do not output analysis, markdown, code fences, "
    "or any text outside the JSON."
)

def build_user_prompt(judgment_text: str) -> str:
    return f"""Read the court judgment carefully and extract structured legal information. Infer every field directly from the judgment text. Do not hallucinate facts or fabricate findings.

Return a JSON object with exactly these keys: case_type, subject, object, objective_aspect, subjective_aspect, reasoning.

FIELD DEFINITIONS AND OUTPUT FORMAT

- case_type:
  One of ["Criminal", "Civil", "Constitutional", "Administrative"].

- subject:
  The primary legal actor. Use the ACTUAL NAME(S) of the party as mentioned in the judgment.
  Do NOT write generic labels like "Accused" or "Defendant" alone.
  Examples from real cases:
    Criminal → "Dinesh Yadav @ Dinesh Kumar And 7 Others"
    Civil → "Ram Kumar Singh"
    Constitutional → "Union of India"
    Administrative → "Municipal Corporation of Delhi"

- object:
  The affected party or opposing party. Use a ROLE-BASED LABEL that describes their relationship to the case.
  Examples from real cases:
    Criminal → "Wife/Complainant" or "Complainant/Informant" or "State/Society"
    Civil → "Defendant/Respondent"
    Constitutional → "State Respondent"
    Administrative → "Administrative Authority"

- objective_aspect:
  The observable legal acts, charges, or disputes. Use SEMICOLON-SEPARATED short legal terms.
  Do NOT write full sentences. Do NOT write paragraphs.
  Examples from real cases:
    Criminal → "Assault; Bigamy; Cruelty; Intentional Insult; Criminal Intimidation"
    Criminal → "Rioting; Rioting With Deadly Weapon; Assault; Attempt To Commit Culpable Homicide"
    Criminal → "Kidnapping Or Abduction"
    Civil → "Breach Of Contract; Non-Payment Of Dues"
    Constitutional → "Violation Of Article 14; Arbitrary State Action"

- subjective_aspect:
  The intent, motive, or legal claim. Use a CONCISE LABEL (2-5 words).
  Do NOT write full sentences.
  Examples from real cases:
    Criminal → "Dowry Demand"
    Criminal → "Intent/Knowledge To Cause Death Or Serious Harm"
    Civil → "Recovery Of Possession"
    Constitutional → "Enforcement Of Fundamental Rights"

- reasoning:
  A concise 3-5 sentence summary. Follow this specific structure:
  Start with "The legal issue was whether..." or "The court considered..."
  Then explain what the court found and decided.
  Focus on judicial findings and the final outcome.
  Do NOT repeat FIR allegations, pleadings, or party arguments.

STRICT RULES:
- If a field cannot be determined, use "Unknown".
- Return ONLY the JSON object.
- No markdown, no code blocks, no explanation outside JSON.

JUDGMENT:
\'\'\'{judgment_text}\'\'\'"""

## 7. Inference

In [9]:
def generate(judgment_text: str):
    """Run one generation. Returns (response_text, n_new_tokens, elapsed_seconds)."""
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + build_user_prompt(str(judgment_text))},
    ]
    # Gemma-2 chat template: Gemma doesn't use a separate system role in its
    # chat template — the system instruction is prepended to the user message.
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=MAX_INPUT_TOKENS,
    ).to(model.device)

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = time.time() - t0

    gen_ids = outputs[0][inputs.input_ids.shape[1]:]
    n_new_tokens = int(gen_ids.shape[0])
    response = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return response, n_new_tokens, elapsed

## 8. JSON Parsing

Strips code fences and leading prose, extracts the first balanced `{...}` block, normalizes to schema.

In [10]:
def _extract_json_block(raw: str):
    """Return the first balanced {...} substring, or None."""
    s = raw.strip()

    # Strip markdown code fences
    s = re.sub(r"```(?:json)?\s*", "", s)
    s = re.sub(r"```", "", s)
    s = s.strip()

    # Find first {
    start = s.find("{")
    if start == -1:
        return None

    # Find balanced closing }
    depth = 0
    for i in range(start, len(s)):
        if s[i] == "{":
            depth += 1
        elif s[i] == "}":
            depth -= 1
            if depth == 0:
                return s[start:i + 1]
    return None


def parse_json(raw: str):
    """Parse model output into the fixed schema. Returns dict or None on failure."""
    block = _extract_json_block(raw)
    if block is None:
        return None

    # Handle trailing commas (occasional Gemma quirk)
    block_clean = re.sub(r",\s*}", "}", block)

    data = None
    for candidate in [block, block_clean]:
        try:
            data = json.loads(candidate)
            if isinstance(data, dict):
                break
        except json.JSONDecodeError:
            data = None

    if not isinstance(data, dict):
        return None

    out = {}
    for k in SCHEMA_KEYS:
        v = data.get(k, "Unknown")
        if v is None or (isinstance(v, str) and not v.strip()):
            v = "Unknown"
        out[k] = v if isinstance(v, str) else json.dumps(v, ensure_ascii=False)

    # Normalize case_type
    ct = out["case_type"].strip().lower()
    out["case_type"] = next((c for c in CASE_TYPES if c.lower() in ct), out["case_type"])
    return out

## 9. Error Handling, Retries & Per-Record Extraction

In [11]:
def extract_one(judgment_text: str):
    """Robust single-record extraction with retries."""
    last_raw, last_err = "", None
    total_tokens, total_time = 0, 0.0

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw, n_tok, elapsed = generate(judgment_text)
            last_raw = raw
            total_tokens += n_tok
            total_time += elapsed

            parsed = parse_json(raw)
            if parsed is not None:
                parsed.update({
                    "raw_output": raw, "parse_ok": True, "attempts": attempt,
                    "error": "", "n_tokens": total_tokens,
                    "gen_time": round(total_time, 3),
                })
                return parsed
            last_err = "JSON parse failed"
        except Exception as e:
            last_err = f"{type(e).__name__}: {e}"
            last_raw = last_raw or traceback.format_exc()

    # All attempts failed
    failed = {k: "Unknown" for k in SCHEMA_KEYS}
    failed.update({
        "raw_output": last_raw, "parse_ok": False, "attempts": MAX_RETRIES,
        "error": last_err or "unknown error",
        "n_tokens": total_tokens, "gen_time": round(total_time, 3),
    })
    return failed

### Quick check on one record

In [12]:
_demo = extract_one(work_df.iloc[0][TEXT_COLUMN])
print("parse_ok:", _demo["parse_ok"], "| attempts:", _demo["attempts"])
print(json.dumps({k: _demo[k] for k in SCHEMA_KEYS}, indent=2, ensure_ascii=False))

parse_ok: True | attempts: 1
{
  "case_type": "Criminal",
  "subject": "Dinesh Yadav @ Dinesh Kumar And 7 Others",
  "object": "State/Society",
  "objective_aspect": "Nullifying FIR; Quashing Chargesheet; Dismissing Criminal Case",
  "subjective_aspect": "Settlement Agreement",
  "reasoning": "The legal issue was whether the criminal case should be dismissed based on a settlement agreement. The court found that a settlement agreement had been reached between the parties and the chargesheet had been validated by the trial court. Accordingly, the court quashed the FIR, chargesheet, and all proceedings in the criminal case."
}


## 10. Batch Inference — Progress Tracking & Performance Monitoring

In [13]:
records, failed_log = [], []
run_start = time.time()

pbar = tqdm(work_df.iterrows(), total=len(work_df), desc="Extracting")
for idx, row in pbar:
    res = extract_one(row[TEXT_COLUMN])
    res["case_id"]  = row.get("case_id", idx)
    res["language"] = row.get("language", "unknown")
    records.append(res)

    if not res["parse_ok"]:
        failed_log.append({"case_id": res["case_id"], "error": res["error"]})

    tps = (res["n_tokens"] / res["gen_time"]) if res["gen_time"] > 0 else 0.0
    pbar.set_postfix({
        "ok": res["parse_ok"],
        "t/judg": f"{res['gen_time']:.1f}s",
        "tok": res["n_tokens"],
        "tok/s": f"{tps:.1f}",
    })

total_runtime = time.time() - run_start
n_ok   = sum(r["parse_ok"] for r in records)
sum_tok = sum(r["n_tokens"] for r in records)

print("\n================ RUN SUMMARY ================")
print(f"Model             : {MODEL_NAME}")
print(f"Records processed : {len(records)}")
print(f"Parsed OK         : {n_ok}")
print(f"Failed            : {len(records) - n_ok}")
print(f"Total tokens      : {sum_tok}")
print(f"Total runtime     : {total_runtime:.1f}s")
if records:
    print(f"Avg time/judgment : {total_runtime / len(records):.1f}s")
if total_runtime > 0:
    print(f"Overall tokens/sec: {sum_tok / total_runtime:.1f}")
if failed_log:
    print("Failed records    :", failed_log)

Extracting:   0%|          | 0/4 [00:00<?, ?it/s]


================ RUN SUMMARY ================
Model             : google/gemma-2-9b-it
Records processed : 4
Parsed OK         : 4
Failed            : 0
Total tokens      : 586
Total runtime     : 48.5s
Avg time/judgment : 12.1s
Overall tokens/sec: 12.1


## 11. Result Saving

In [14]:
results_df = pd.DataFrame(records)

append_cols = SCHEMA_KEYS + ["raw_output", "parse_ok", "attempts", "error"]
results_slim = results_df[["case_id"] + append_cols].copy()

base = (work_df if SAMPLE_MODE else df).copy()
base["case_id"] = base["case_id"].astype(str)
results_slim["case_id"] = results_slim["case_id"].astype(str)

base = base.drop(columns=[c for c in append_cols if c in base.columns], errors="ignore")

final_df = base.merge(results_slim, on="case_id", how="left")
final_df.to_excel(OUTPUT_PATH, index=False)
print(f"Saved {len(final_df)} rows -> {OUTPUT_PATH}")

final_df[["case_id"] + SCHEMA_KEYS].head(10)

Saved 4 rows -> indilex_extracted_output_Gemma.xlsx


/tmp/ipykernel_28625/3445876306.py:13: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  final_df.to_excel(OUTPUT_PATH, index=False)


,case_id,case_type,subject,object,objective_aspect,subjective_aspect,reasoning
0,ALL_CRIM_000001,Criminal,Dinesh Yadav @ Dinesh Kumar And 7 Others,State/Society,Nullification Of FIR; Quashing Of Chargesheet;...,Settlement Between Parties,The legal issue was whether the criminal case ...
1,ALL_CRIM_000002,Criminal,Kamlesh Jaiswal Alias Monu Jaiswal And Another,State/Society,"Section 18 14, 572: 303. 41, 44 IPC and Sectio...",False Implication,The legal issue was whether the applicants wer...
2,ALL_CRIM_000003,Criminal,Bahal Khan And 4 Others,State/Society,Assault; Grievous Hurt; Rioting; Weapon Use,Settlement Agreement,The legal issue was whether the criminal case ...
3,ALL_CRIM_000004,Criminal,Banshi Lal,State/Society,Kidnapping Or Abduction; False Complaint,Challenge to Final Report,The legal issue was whether the impugned order...


## 12. Full Dataset

Set `SAMPLE_MODE = False` in Config cell, re-run from Dataset Loading downward.